Linear and input multi-head projection

In [2]:
import torch
import torch.nn as nn


In [3]:
# nn.Linear applies an affine linear transformation: y = xW^T + b
#
# Key Components:
# 1. Weight matrix W: Shape(out_features, in_features)
# 2. Bias vector b: Shape(out_features,) - optional
# 3. Forward pass: output = input @ W^T + b
#
# Weights is stored as (out_features, in_features) but transposed during forward pass
# This allows efficient matrix multiplication: (*, in_features) @ (in_features, out_features) -> output shape (*, out_features)

# Crate a linear layer: in_features=3, out_features=2
linear = nn.Linear(in_features=3, out_features=2, bias=True)
print("Linear Layer Configuration:")
print(f"  Input features: {linear.in_features}")
print(f"  Output features: {linear.out_features}")
print(f"  Weight shape: {linear.weight.shape}")
print(f"  Weight matrix:\n{linear.weight}")
print(f"  Bias shape: {linear.bias.shape if linear.bias is not None else 'None'}")
print(f"  Bias vector: {linear.bias}")

Linear Layer Configuration:
  Input features: 3
  Output features: 2
  Weight shape: torch.Size([2, 3])
  Weight matrix:
Parameter containing:
tensor([[ 0.4124, -0.4967,  0.1605],
        [ 0.3216, -0.5771,  0.4085]], requires_grad=True)
  Bias shape: torch.Size([2])
  Bias vector: Parameter containing:
tensor([0.0144, 0.1947], requires_grad=True)


In [5]:
# input batch of 4 samples, each with 3 features
x = torch.tensor([[1.0,2.0,3.0], [4.0,5.0,6.0], [7.0,8.0,9.0], [10.0,11.0,12.0]], dtype=torch.float32)
print(f"Input shape: {x.shape}\n Input:\n{x}")

# Forward pass
output = linear(x)
print(f"Output shape: {output.shape}\n Output:\n{output}")

Input shape: torch.Size([4, 3])
 Input:
tensor([[ 1.,  2.,  3.],
        [ 4.,  5.,  6.],
        [ 7.,  8.,  9.],
        [10., 11., 12.]])
Output shape: torch.Size([4, 2])
 Output:
tensor([[-0.0850,  0.5876],
        [ 0.1437,  1.0465],
        [ 0.3724,  1.5054],
        [ 0.6012,  1.9644]], grad_fn=<AddmmBackward0>)


In [11]:
# Create linear layer with fixed weights for clarity
linear_fixed = nn.Linear(in_features=3, out_features=2, bias=True)

with torch.no_grad():
    # store in (output_features, in_features) shape
    linear_fixed.weight = nn.Parameter(torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]]))
    linear_fixed.bias = nn.Parameter(torch.tensor([10.0, 20.0]))

print("Linear Layer Configuration:")
print(f"  Input features: {linear_fixed.in_features}")
print(f"  Output features: {linear_fixed.out_features}")
print(f"  Weight shape: {linear_fixed.weight.shape}")
print(f"  Weight matrix:\n{linear_fixed.weight}")
print(f"  Bias shape: {linear_fixed.bias.shape if linear_fixed.bias is not None else 'None'}")
print(f"  Bias vector: {linear_fixed.bias}")

x = torch.tensor([[1.0, 2.0, 3.0]])
print(f"x shape: {x.shape}\n{x}")

output_auto = linear_fixed(x)
print(f"Output using nn.Linear:\n{output_auto}")

W_T = linear_fixed.weight.T
print(f"\nWeight transpose shape: {W_T.shape}\n{W_T}")

output_manual = torch.matmul(x, W_T) + linear_fixed.bias
print(f"Output using manual calculation:\n{output_auto}")

print(f"Results match: {torch.allclose(output_auto, output_manual)}")

Linear Layer Configuration:
  Input features: 3
  Output features: 2
  Weight shape: torch.Size([2, 3])
  Weight matrix:
Parameter containing:
tensor([[1., 2., 3.],
        [4., 5., 6.]], requires_grad=True)
  Bias shape: torch.Size([2])
  Bias vector: Parameter containing:
tensor([10., 20.], requires_grad=True)
x shape: torch.Size([1, 3])
tensor([[1., 2., 3.]])
Output using nn.Linear:
tensor([[24., 52.]], grad_fn=<AddmmBackward0>)

Weight transpose shape: torch.Size([3, 2])
tensor([[1., 4.],
        [2., 5.],
        [3., 6.]], grad_fn=<PermuteBackward0>)
Output using manual calculation:
tensor([[24., 52.]], grad_fn=<AddmmBackward0>)
Results match: True


In [17]:
# Simulating Q, K, V projections in attention mechanism
d_model = 512
h = 8
d_k = d_v = d_model // h

# Linear projections for Q, K, V
W_q = nn.Linear(d_model, d_model, bias=False)
W_k = nn.Linear(d_model, d_model, bias=False)
W_v = nn.Linear(d_model, d_model, bias=False)

print(f"MHA linear projections:")
print(f"d_model: {d_model}, h(heads): {h}, d_k(per head): {d_k}, d_v: {d_v}")
print(f"W_q/W_k/W_v shape: {W_q.weight.shape}")

# Input: batch_size=2, seq_len=10, d_model=512
batch_size, seq_len = 2, 10
x = torch.randn(batch_size, seq_len, d_model)
print(f"x shape: {x.shape}")

Q = W_q(x)
K = W_k(x)
V = W_v(x)
print(f"Q/K/V shape after projection: {Q.shape}")

# Reshape for multi-head attention: (batch_size, seq_len, d_model) -> (batch_size, seq_len, h, d_k)
Q = Q.view(batch_size, seq_len, h, d_k).transpose(1, 2)  # (batch_size, h, seq_len, d_k)
K = K.view(batch_size, seq_len, h, d_k).transpose(1, 2)  # (batch_size, h, seq_len, d_k)
V = V.view(batch_size, seq_len, h, d_k).transpose(1, 2)  # (batch_size, h, seq_len, d_k)

print(f"Q/K/V shape after reshape for multi-head: {Q.shape}")
print("Explanation:")
print("1. Linear layers project input from d_model to d_model")
print("2. Reshape splits d_model into (h × d_k)")
print("3. Each head operates on d_k=64 dimensions")
print("4. This allows parallel attention computation across heads")


MHA linear projections:
d_model: 512, h(heads): 8, d_k(per head): 64, d_v: 64
W_q/W_k/W_v shape: torch.Size([512, 512])
x shape: torch.Size([2, 10, 512])
Q/K/V shape after projection: torch.Size([2, 10, 512])
Q/K/V shape after reshape for multi-head: torch.Size([2, 8, 10, 64])
Explanation:
1. Linear layers project input from d_model to d_model
2. Reshape splits d_model into (h × d_k)
3. Each head operates on d_k=64 dimensions
4. This allows parallel attention computation across heads
